In [1]:
from src import ppt_executor, ppt_reader, openai_api, prompt_factor, dataset, api_selection, utils, api_doc
import openai
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
instruction = "Fill the title with  \"Contents\" , and move it on the top of the picture."

In [3]:
planning_prompt = prompt_factor.query_decomposition_prompt.format(instruction)
planning_reply = openai_api.query_azure_openai(planning_prompt, model="gpt-4o-mini").strip()
decomposed = planning_reply.split('\n')
decomposed = [d.replace('</d>','') for d in decomposed if (d != '</d>') and (d != '<d>')]
print(f"{instruction}->{decomposed}")

Fill the title with  "Contents" , and move it on the top of the picture.->['Fill the title with "Contents".', 'Move the title to the top of the picture.']


In [4]:
chat_history = [
    {"role":"system", "content":"You are an expert in PowerPoint."},
    {"role":"user", "content":"Set the title to 'cheese'."}
]

toolkit = [
    {
            'type': 'function',
            'function': {
                'name': "Insert_Text",
                'description': "insert or name text, title, content, table, textbox with given text",
                'parameters': {
                    'type': 'object',
                    'properties': {
                        "text": {'description': 'Text to be inserted', 'type': 'string'}
                    }
                },
                'required': ["text"]
            }
        }
]

In [5]:
tool_calls = openai_api.query_openai("Set the title to 'cheese'.", "gpt-4o-mini", toolkit)
tool_calls

[<OpenAIObject id=call_yDxJWPbLfryK7sF5G3qumSj3 at 0x27affe0e580> JSON: {
   "function": {
     "arguments": "{\"text\":\"cheese\"}",
     "name": "Insert_Text"
   },
   "id": "call_yDxJWPbLfryK7sF5G3qumSj3",
   "type": "function"
 }]

In [7]:
import json
def parse_api2(tool_calls):
    apis = []
    for tool_call in tool_calls:
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)
        args_str = ', '.join(f'{key}={value!r}' for key, value in func_args.items())

        api_str = f"{func_name}({args_str})"
        apis.append(api_str)
    
    #print(apis)
    return apis

parse_api2(tool_calls)

["Insert_Text(text='cheese')"]